# EoMT trên BTXRD (Colab / Kaggle): phát hiện + phân đoạn u xương và phân loại lành / ác trong một lần chạy

Notebook này huấn luyện **EoMT (Encoder-only Mask Transformer, CVPR 2025)** của dự án trên **toàn bộ** bộ ảnh X-quang **BTXRD**, gồm cả ảnh **có u** (có mask) và ảnh **bình thường** (không có mask), theo đúng quy trình của repo: `LightningDataModule` trong `datasets/`, `LightningModule` trong `training/`, file cấu hình YAML trong `configs/`, huấn luyện bằng `python main.py fit -c <config>`.

**Cách đặt bài toán:** instance segmentation với 2 lớp mask `benign` / `malignant`, cộng lớp **"no object"** có sẵn của EoMT.
- Ảnh có u: mỗi polygon khối u trong file LabelMe là một mask, nhãn của mask là loại lành/ác của ảnh.
- Ảnh bình thường: target **không có mask nào**, nên mọi query của EoMT học dự đoán "no object".
- Suy luận, **một lần forward duy nhất** cho mỗi ảnh:
  - điểm u = max theo query của P(benign) + P(malignant); **có u** nếu điểm u > 0.5 (phát hiện);
  - **vùng u** = hợp các mask của những query vượt ngưỡng, rỗng nếu không phát hiện u (phân đoạn);
  - **lành / ác** theo query tin cậy nhất (phân loại).

**Dữ liệu:** 3.726 ảnh (1.879 bình thường, 1.515 u lành, 332 u ác), chia **train : val : test = 80 : 15 : 5**, phân tầng theo loại ảnh bằng `tools/split_btxrd.py` (seed 42):

| | normal | benign | malignant | tổng |
|---|---:|---:|---:|---:|
| train | 1503 | 1212 | 265 | 2980 |
| val | 282 | 227 | 50 | 559 |
| test | 94 | 76 | 17 | 187 |

**Chỉ số đánh giá**
- Phát hiện u (u là lớp dương, trên mọi ảnh): **AUROC**, Sensitivity, Specificity, Balanced accuracy, F1.
- Phân đoạn: **Dice**, **IoU** trên ảnh có u; Dice trên toàn bộ ảnh (ảnh bình thường dự đoán rỗng tính Dice = 1); mask **AP** (COCO-style) như dự án gốc.
- Phân loại lành / ác (ác là lớp dương, trên ảnh có u): **AUROC**, Sensitivity, Specificity, Balanced accuracy.
- Phân loại 3 lớp theo ảnh (bình thường / lành / ác): Accuracy, Balanced accuracy.
- Chọn checkpoint tốt nhất theo `val_score = (AUROC phát hiện + Dice + AUROC lành/ác) / 3`.

**Thiết lập Google Colab**
1. *Runtime → Change runtime type*: **GPU** (T4 là đủ; L4 / A100 nhanh hơn).
2. **Dữ liệu tự tải từ Kaggle**: dataset [`nguyetanhmai/btxrd-full-split`](https://www.kaggle.com/datasets/nguyetanhmai/btxrd-full-split) (biến `KAGGLE_DATASET`), tải bằng `kagglehub` vào ổ đĩa của Colab. Không cần đưa dữ liệu lên Drive.
   - Dataset công khai thì tải được ngay. Nếu dataset để **private** (hoặc Kaggle báo cần đăng nhập): vào biểu tượng 🔑 *Secrets* của Colab, thêm `KAGGLE_USERNAME` và `KAGGLE_KEY` (lấy từ *Kaggle → Settings → API → Create New Token*), bật *Notebook access*.
3. Chạy *Runtime → Run all*, cho phép notebook truy cập Google Drive khi được hỏi (chỉ để lưu kết quả; đặt `SAVE_TO_DRIVE = False` nếu không cần).
4. Kết quả (checkpoint, log, trọng số, dự đoán) được lưu ở `MyDrive/btxrd_eomt/`. Nếu Colab ngắt kết nối giữa chừng, chỉ cần *Run all* lại: notebook tự **tiếp tục từ `last.ckpt`**.
5. (Tùy chọn) *Secrets*: thêm `WANDB_API_KEY` để log lên Weights & Biases.

> Colab thường chỉ có **1 GPU**, nên tổng batch là 4 (Kaggle 2×T4 là 8), số bước gấp đôi: 40 epoch ≈ 29.800 bước, cỡ 5–7 giờ trên T4. Nếu phiên Colab ngắn, giảm `EPOCHS` hoặc chạy nhiều phiên nhờ cơ chế resume.

**Thiết lập Kaggle**
1. *Settings → Accelerator*: **GPU T4 x2** (khuyến nghị; P100 không tương thích với các bản PyTorch mới của Kaggle).
2. *Settings → Internet*: **On** (để clone repo, cài thư viện, tải trọng số DINOv2 / EoMT COCO).
3. *Add Input*: dataset `nguyetanhmai/btxrd-full-split`, hoặc một trong hai dạng dữ liệu sau (notebook tự tìm trong `/kaggle/input`):
   - **Đã chia** (thư mục `BTXRD_full_split` tạo bởi `tools/split_btxrd.py`):
     ```
     BTXRD_full_split/
     ├── split.csv                  # image_id,split,type (normal|benign|malignant)
     ├── train/{images,Annotations}/   # Annotations chỉ có cho ảnh có u
     ├── val/{images,Annotations}/
     └── test/{images,Annotations}/
     ```
   - **Bản gốc phẳng** (`images/`, `Annotations/`, `dataset.xlsx`): notebook tự chia 80:15:5 bằng cùng script và cùng seed, nên ra đúng cách chia như trên.
4. (Tùy chọn) *Add-ons → Secrets*: thêm `WANDB_API_KEY` để log lên Weights & Biases như dự án gốc. Nếu không có, notebook chỉ log ra CSV.

## 1. Cấu hình

In [ ]:
import os
import sys
import csv
import json
import glob
import math
import shutil
import subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()
try:
    import google.colab  # noqa: F401

    ON_COLAB = True
except ImportError:
    ON_COLAB = False
ON_CLOUD = ON_KAGGLE or ON_COLAB

SAVE_TO_DRIVE = True  # Colab: keep checkpoints / logs / results on Google Drive (survive a disconnect)
if ON_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

# WORK_DIR: fast local disk (repo, data, image cache). OUT_DIR: results that must be kept.
WORK_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB else Path.cwd()
OUT_DIR = Path("/content/drive/MyDrive/btxrd_eomt") if ON_COLAB and SAVE_TO_DRIVE else WORK_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Source code ---
REPO_URL = "https://github.com/67htb5vn/eomt.git"  # fork of tue-mps/eomt, with the BTXRD files
REPO_BRANCH = "master"
REPO_DIR = WORK_DIR / "eomt" if ON_CLOUD else Path.cwd()

# --- Data ---
KAGGLE_DATASET = "nguyetanhmai/btxrd-full-split"  # downloaded with kagglehub when not running on Kaggle
DATA_ROOT = None  # None = auto-detect (Kaggle input / KAGGLE_DATASET); or a folder / .zip to use instead
SPLIT_RATIOS = (0.80, 0.15, 0.05)  # train : val : test, used only when the input is not split yet
SPLIT_SEED = 42
CACHE_DIR = WORK_DIR / "btxrd_cache"  # resized copy of the dataset (faster loading)
CACHE_MAX_SIDE = 1024  # longest image side after caching

# --- Model / training ---
BASE_CONFIG = "configs/dinov2/btxrd/instance/eomt_base_640.yaml"
RUN_NAME = "btxrd_instance_eomt_base_640"
INIT_FROM_COCO = True  # start from EoMT-B trained on COCO panoptic (recommended)
COCO_CKPT_REPO = "tue-mps/coco_panoptic_eomt_base_640_2x"
IMG_SIZE = 640  # must stay 640 when INIT_FROM_COCO=True
EPOCHS = 40
BATCH_SIZE_PER_GPU = 4  # fits a 16 GB T4; Kaggle 2 x T4 = total batch 8, Colab 1 GPU = 4
NUM_WORKERS = min(4, os.cpu_count() or 2)  # Colab usually has 2 CPUs, Kaggle 4
LR = 1e-4
BALANCED_SAMPLING = True  # sample normal / benign / malignant images equally
TUMOR_SCORE_THRESH = 0.5  # image has a tumor if the best query's tumor score is above this
COMPILE = False  # torch.compile; off by default for stability on Kaggle GPUs
SEED = 0
USE_WANDB = True  # only used if a WANDB_API_KEY secret is available

SMOKE_TEST = False  # True = 1 short epoch to check the pipeline end-to-end
RESUME = True  # continue from last.ckpt in CKPT_DIR if it exists (e.g. after a Colab disconnect)

CKPT_DIR = OUT_DIR / "checkpoints" / RUN_NAME
LOG_DIR = OUT_DIR / "logs"
RUN_CONFIG = OUT_DIR / f"{RUN_NAME}.yaml"
TEST_CONFIG = OUT_DIR / f"{RUN_NAME}_test.yaml"
BEST_WEIGHTS = OUT_DIR / f"{RUN_NAME}_best.bin"
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} | work dir: {WORK_DIR} | results: {OUT_DIR}")

## 2. Cài đặt môi trường

In [ ]:
# Versions pinned as in requirements.txt. torch/torchvision come from the Kaggle / Colab image.
PACKAGES = [
    "lightning==2.5.1.post0",
    "jsonargparse[signatures]==4.38",
    "timm==1.0.15",
    "transformers==4.56.1",
    "torchmetrics==1.7.1",
    "pycocotools==2.0.8",
    "gitignore_parser==0.1.12",
    "wandb==0.19.10",
]
if not ON_KAGGLE:
    PACKAGES.append("kagglehub")  # to download the dataset from Kaggle
if ON_CLOUD:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)

import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
NUM_GPUS = torch.cuda.device_count()

## 3. Lấy mã nguồn dự án và kiểm tra các thành phần BTXRD

Clone repo vào `/kaggle/working/eomt` (Kaggle) hoặc `/content/eomt` (Colab). Trên Kaggle, nếu repo đã được thêm làm Input (chạy offline), notebook sẽ sao chép từ đó.

Notebook **không viết lại** code nào của dự án: model EoMT (`models/`), loss, optimizer, lịch học, đánh giá mask AP (`training/`) và lệnh `main.py` đều là của repo. Phần riêng cho BTXRD cũng nằm trong repo (`datasets/btxrd.py`, `training/mask_classification_btxrd.py`, `configs/dinov2/btxrd/`, `tools/split_btxrd.py`); ô dưới chỉ kiểm tra là chúng có mặt.

In [ ]:
def find_input_repo():
    if not ON_KAGGLE:
        return None
    for pattern in ("*/models/eomt.py", "*/*/models/eomt.py"):
        for p in Path("/kaggle/input").glob(pattern):
            return p.parent.parent
    return None


if ON_CLOUD and not (REPO_DIR / "main.py").exists():
    input_repo = find_input_repo()
    if input_repo is not None:
        shutil.copytree(input_repo, REPO_DIR)
        print("Copied repo from", input_repo)
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True
        )

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# The repo's `datasets` package clashes with Hugging Face `datasets` if that was imported.
for name in [m for m in sys.modules if m == "datasets" or m.startswith("datasets.")]:
    del sys.modules[name]

BTXRD_FILES = [
    "datasets/btxrd.py",  # LightningDataModule: BTXRD images + LabelMe polygons, normal images = 0 mask
    "training/mask_classification_btxrd.py",  # subclass of the project's MaskClassificationInstance
    "configs/dinov2/btxrd/instance/eomt_base_640.yaml",  # EoMT-B (DINOv2) config
    "tools/split_btxrd.py",  # 80:15:5 stratified split of the raw release
]
missing = [f for f in BTXRD_FILES if not (REPO_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"{REPO_DIR} has no {missing}. Push these files to {REPO_URL} ({REPO_BRANCH}), "
        "or set REPO_DIR to a copy of the project that contains them."
    )

from models.eomt import EoMT
from training.mask_classification_btxrd import MaskClassificationBTXRD

print("Repo:", REPO_DIR)
print("Model network:", EoMT.__module__ + "." + EoMT.__name__)
print("Lightning module:", " -> ".join(c.__name__ for c in MaskClassificationBTXRD.__mro__[:4]))

## 4. Các thành phần BTXRD trong dự án

Mọi thứ dùng để huấn luyện đều lấy từ repo, theo đúng cách dự án tổ chức một bài toán mới (giống `coco_instance.py` + `mask_classification_instance.py` + file YAML):

| File trong repo | Vai trò | Dùng lại từ dự án gốc |
|---|---|---|
| `models/eomt.py`, `models/vit.py` | Mạng EoMT + backbone DINOv2 | **Nguyên bản, không sửa** |
| `training/mask_classification_loss.py` | Loss Hungarian (mask + dice + phân lớp, có lớp "no object") | **Nguyên bản**; xử lý được ảnh 0 mask nên ảnh bình thường dạy các query dự đoán "no object" |
| `main.py`, `training/lightning_module.py` | CLI, optimizer, LLRD, warmup + poly, mask annealing | **Nguyên bản** |
| `datasets/btxrd.py` | `BTXRDInstance` (LightningDataModule) đọc ảnh, polygon LabelMe, ảnh bình thường có 0 mask | Kế thừa `LightningDataModule`, `Transforms` |
| `training/mask_classification_btxrd.py` | `MaskClassificationBTXRD`: gọi `eval_step` của dự án (mask AP), thêm phần đọc kết quả phát hiện / phân đoạn / lành-ác từ lớp cuối và các chỉ số | Kế thừa `MaskClassificationInstance` |
| `configs/dinov2/btxrd/instance/eomt_base_640.yaml` | Cấu hình EoMT-B 640 | Cùng định dạng các config COCO |
| `tools/split_btxrd.py` | Chia BTXRD gốc 80:15:5 theo loại ảnh | — |

## 5. Dữ liệu BTXRD: tải, chia tập và khám phá

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from PIL import Image

from datasets.btxrd import (
    IMAGE_CLASSES,
    IMAGE_CLASS_TO_ID,
    IMG_SUFFIXES,
    load_tumor_shapes,
)


EXTRACT_DIR = WORK_DIR / "btxrd_data"  # where a dataset zip is extracted
# Never pick the 20-image sample shipped in the repo, or our own cache
EXCLUDED = [REPO_DIR, CACHE_DIR]


def search(bases, name, max_depth):
    for base in bases:
        for depth in range(max_depth + 1):
            for hit in sorted(base.glob("*/" * depth + name)):
                if not any(hit.is_relative_to(e) for e in EXCLUDED):
                    yield hit


def find_folder(bases, max_depth=3):
    """Folder with split.csv (already split), else folder with images/ + dataset.xlsx."""
    for hit in search(bases, "split.csv", max_depth):
        return hit.parent
    for hit in search(bases, "dataset.xlsx", max_depth):
        if (hit.parent / "images").is_dir():
            return hit.parent
    return None


def extract(zip_path):
    if not (EXTRACT_DIR / ".done").exists():
        print("Extracting", zip_path, "->", EXTRACT_DIR)
        shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
        shutil.unpack_archive(str(zip_path), str(EXTRACT_DIR))
        (EXTRACT_DIR / ".done").touch()
    root = find_folder([EXTRACT_DIR])
    if root is None:
        raise FileNotFoundError(f"No split.csv or images/ + dataset.xlsx inside {zip_path}")
    return root


def download_kaggle_dataset(handle):
    """Download a Kaggle dataset with kagglehub (Colab / local). Credentials, if needed,
    come from Colab Secrets KAGGLE_USERNAME / KAGGLE_KEY or ~/.kaggle/kaggle.json."""
    if ON_COLAB and not os.environ.get("KAGGLE_KEY"):
        from google.colab import userdata

        for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            try:
                os.environ[name] = userdata.get(name)
            except Exception:
                pass
    import kagglehub

    print("Downloading Kaggle dataset", handle)
    path = Path(kagglehub.dataset_download(handle))
    print("Downloaded to", path)
    return path


def find_data_root():
    if DATA_ROOT is not None:
        root = Path(DATA_ROOT)
        return extract(root) if root.suffix.lower() == ".zip" else root
    if ON_KAGGLE:  # dataset added with "Add Input"
        root = find_folder([Path("/kaggle/input")])
        if root is not None:
            return root
    elif KAGGLE_DATASET:
        path = download_kaggle_dataset(KAGGLE_DATASET)
        root = find_folder([path], max_depth=4)
        if root is not None:
            return root
        zips = sorted(path.rglob("*.zip"))
        if zips:
            return extract(zips[0])
        raise FileNotFoundError(f"No split.csv or images/ + dataset.xlsx in Kaggle dataset {KAGGLE_DATASET}")
    else:
        root = find_folder([REPO_DIR.parent, REPO_DIR])
        if root is not None:
            return root
    raise FileNotFoundError("No BTXRD data found; set KAGGLE_DATASET or DATA_ROOT")


SRC_ROOT = find_data_root()
print("Dataset:", SRC_ROOT)

if (SRC_ROOT / "split.csv").exists():
    split_df = pd.read_csv(SRC_ROOT / "split.csv")
else:
    from tools.split_btxrd import make_split

    split_df = pd.DataFrame(make_split(SRC_ROOT, SPLIT_RATIOS, SPLIT_SEED))
    print(f"No split.csv: stratified split {SPLIT_RATIOS} with seed {SPLIT_SEED}")

split_df["type"] = split_df["type"].str.strip().str.lower()
if "normal" not in set(split_df["type"]):
    print("WARNING: no normal image in this dataset, the model will not learn tumor detection")

counts = pd.crosstab(split_df["split"], split_df["type"], margins=True, margins_name="total")
counts = counts.reindex(index=["train", "val", "test", "total"],
                        columns=[*IMAGE_CLASSES, "total"], fill_value=0)
display(counts)
display((100 * counts.div(counts.loc["total", "total"])).round(1).rename(columns=lambda c: f"{c} %"))


def list_source_items(root):
    """(split, image path, annotation path) for both supported layouts."""
    items = []
    split_dirs = [s for s in ("train", "val", "test") if (root / s / "images").is_dir()]
    if split_dirs:
        for s in split_dirs:
            for img in sorted((root / s / "images").iterdir()):
                if img.suffix.lower() in IMG_SUFFIXES:
                    items.append((s, img, root / s / "Annotations" / f"{img.stem}.json"))
    else:  # flat layout: images/ + Annotations/, split taken from split_df
        split_of = {Path(i).stem: s for i, s in zip(split_df["image_id"], split_df["split"])}
        for img in sorted((root / "images").iterdir()):
            s = split_of.get(img.stem)
            if s and img.suffix.lower() in IMG_SUFFIXES:
                items.append((s, img, root / "Annotations" / f"{img.stem}.json"))
    return items


source_items = list_source_items(SRC_ROOT)
image_types = {
    Path(i).stem: IMAGE_CLASS_TO_ID[t]
    for i, t in zip(split_df["image_id"], split_df["type"])
    if t in IMAGE_CLASS_TO_ID
}

rows = []
for split, img_path, ann_path in source_items:
    polygons, h, w = load_tumor_shapes(ann_path) if ann_path.exists() else ([], None, None)
    image_type = IMAGE_CLASSES[image_types[img_path.stem]] if img_path.stem in image_types else None
    rows.append(
        dict(
            split=split,
            image=img_path.stem,
            type=image_type,
            subtype=polygons[0][1] if polygons else "-",
            n_tumors=len(polygons),
            height=h,
            width=w,
        )
    )
meta_df = pd.DataFrame(rows)
tumor_df = meta_df[meta_df.type.isin(["benign", "malignant"])]
print(f"{len(meta_df)} images | normal: {(meta_df.type == 'normal').sum()} | "
      f"tumor: {len(tumor_df)} | tumor without polygon: {(tumor_df.n_tumors == 0).sum()}")
display(meta_df.groupby(["type", "subtype"]).size().rename("images").to_frame())
display(tumor_df[["n_tumors", "height", "width"]].describe().round(1))

In [ ]:
def show_samples(df, n_per_class=2):
    picks = pd.concat(
        [g.sample(min(n_per_class, len(g)), random_state=SEED) for _, g in df.dropna(subset=["type"]).groupby("type")]
    )
    fig, axes = plt.subplots(1, len(picks), figsize=(4 * len(picks), 5))
    for ax, (_, r) in zip(np.atleast_1d(axes), picks.iterrows()):
        split, img_path, ann_path = next(it for it in source_items if it[1].stem == r.image)
        with Image.open(img_path) as im:
            ax.imshow(im.convert("L"), cmap="gray")
        if ann_path.exists() and r.type != "normal":
            color = "tab:red" if r.type == "malignant" else "tab:green"
            for points, _ in load_tumor_shapes(ann_path)[0]:
                ax.add_patch(MplPolygon(points, closed=True, fill=False, ec=color, lw=2))
        ax.set_title(f"{r.image}\n{r.type} / {r.subtype}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_samples(meta_df[(meta_df.type == "normal") | (meta_df.n_tumors > 0)])

## 6. Tiền xử lý: cache ảnh đã thu nhỏ

Ảnh X-quang gốc khoảng 2000–3000 px, giải mã JPEG mỗi bước huấn luyện sẽ rất chậm. Ta thu nhỏ một lần sao cho cạnh dài ≤ `CACHE_MAX_SIDE` (vẫn đủ lớn cho scale jitter ở 640) và đổi tọa độ polygon theo cùng tỉ lệ. Ảnh bình thường không có annotation nên chỉ cần thu nhỏ ảnh. Thư mục cache có cấu trúc `train/val/test` và `split.csv` giống hệt đầu ra của `tools/split_btxrd.py`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

DONE_MARKER = CACHE_DIR / f".done_{CACHE_MAX_SIDE}"


def cache_one(item):
    split, img_path, ann_path = item
    out_img = CACHE_DIR / split / "images" / f"{img_path.stem}.jpeg"
    out_ann = CACHE_DIR / split / "Annotations" / f"{img_path.stem}.json"

    with Image.open(img_path) as im:
        orig_w, orig_h = im.size
        im.draft("RGB", (CACHE_MAX_SIDE, CACHE_MAX_SIDE))  # fast JPEG downscale
        im = im.convert("L" if im.mode in ("L", "I;16", "I") else "RGB")
        im.thumbnail((CACHE_MAX_SIDE, CACHE_MAX_SIDE), Image.LANCZOS)
        new_w, new_h = im.size
        im.save(out_img, quality=95)

    if ann_path.exists():  # tumor images only
        with open(ann_path, encoding="utf-8") as f:
            ann = json.load(f)
        # Polygons are relative to the size stored in the json (fallback: real size)
        sx = new_w / (ann.get("imageWidth") or orig_w)
        sy = new_h / (ann.get("imageHeight") or orig_h)
        for shape in ann.get("shapes", []):
            shape["points"] = [[x * sx, y * sy] for x, y in shape["points"]]
        ann.update(imageWidth=new_w, imageHeight=new_h, imageData=None, imagePath=out_img.name)
        with open(out_ann, "w", encoding="utf-8") as f:
            json.dump(ann, f)


if not DONE_MARKER.exists():
    for split in {s for s, _, _ in source_items}:
        (CACHE_DIR / split / "images").mkdir(parents=True, exist_ok=True)
        (CACHE_DIR / split / "Annotations").mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as pool:
        list(pool.map(cache_one, source_items))
    split_df.to_csv(CACHE_DIR / "split.csv", index=False)
    if (SRC_ROOT / "dataset.xlsx").exists():
        shutil.copy(SRC_ROOT / "dataset.xlsx", CACHE_DIR / "dataset.xlsx")
    DONE_MARKER.touch()

for split in ("train", "val", "test"):
    if (CACHE_DIR / split).exists():
        n_img = len(list((CACHE_DIR / split / "images").iterdir()))
        n_ann = len(list((CACHE_DIR / split / "Annotations").iterdir()))
        print(f"{split:5s}: {n_img} images, {n_ann} annotations (tumor images)")

## 7. Kiểm tra DataModule

Khởi tạo `BTXRDInstance` như `main.py` sẽ làm, rồi xem vài mẫu train sau augmentation (scale jitter, flip, crop, color jitter), gồm cả ảnh bình thường (0 mask), và một mẫu val.

In [ ]:
from collections import Counter
from datasets.btxrd import BTXRDInstance

data = BTXRDInstance(
    path=str(CACHE_DIR),
    batch_size=BATCH_SIZE_PER_GPU,
    num_workers=0,
    img_size=(IMG_SIZE, IMG_SIZE),
    check_empty_targets=False,
).setup()

for name in ("train", "val", "test"):
    ds = getattr(data, f"{name}_dataset")
    if ds is not None:
        counts = Counter(IMAGE_CLASSES[c] for c in ds.class_ids)
        print(f"{name:5s}: {len(ds):5d} images  {dict(sorted(counts.items()))}")

MASK_COLORS = {0: np.array([0.1, 0.8, 0.2]), 1: np.array([0.9, 0.1, 0.1])}  # benign, malignant


def overlay(ax, img, masks, labels, title):
    ax.imshow(img.permute(1, 2, 0).numpy())
    for m, l in zip(masks, labels):
        rgba = np.zeros((*m.shape, 4))
        rgba[m.numpy().astype(bool)] = [*MASK_COLORS[int(l)], 0.4]
        ax.imshow(rgba)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


train_ids = data.train_dataset.class_ids
picks = []  # two normal images, one benign, one malignant
for c in (0, 0, 1, 2):
    cand = [i for i, x in enumerate(train_ids) if x == c and i not in picks]
    if cand:
        picks.append(cand[0])
val_pick = next(i for i, c in enumerate(data.val_dataset.class_ids) if c > 0)

fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
for ax, idx in zip(axes[:4], picks):
    img, tgt = data.train_dataset[idx]
    overlay(ax, img, tgt["masks"], tgt["labels"],
            f"train (aug) {tuple(img.shape[-2:])}\n{IMAGE_CLASSES[int(tgt['image_label'])]}, {len(tgt['masks'])} mask(s)")
img, tgt = data.val_dataset[val_pick]
overlay(axes[4], img, tgt["masks"], tgt["labels"],
        f"val {tuple(img.shape[-2:])}\n{IMAGE_CLASSES[int(tgt['image_label'])]}, {len(tgt['masks'])} mask(s)")
plt.tight_layout()
plt.show()

## 8. Tạo file cấu hình cho lần chạy

Bắt đầu từ `configs/dinov2/btxrd/instance/eomt_base_640.yaml`, cập nhật theo kích thước dữ liệu và phần cứng:
- `attn_mask_annealing_*_steps` và `warmup_steps` co giãn theo tổng số bước, cùng tỉ lệ với config COCO/ADE20K gốc (annealing kết thúc ở khoảng 5/6 quá trình huấn luyện).
- Fine-tune từ checkpoint EoMT-B COCO panoptic (`--model.ckpt_path`, `--model.load_ckpt_class_head False`) như hướng dẫn trong README.
- Nhiều GPU thì dùng DDP; logger là CSV, thêm W&B nếu có `WANDB_API_KEY`.

In [ ]:
import yaml

with open(REPO_DIR / BASE_CONFIG) as f:
    cfg = yaml.safe_load(f)

devices = max(1, NUM_GPUS)
steps_per_epoch = len(data.train_dataset) // (BATCH_SIZE_PER_GPU * devices)
epochs = 1 if SMOKE_TEST else EPOCHS
total_steps = max(1, steps_per_epoch * epochs)

# Mask-annealing schedule, as in the COCO/ADE20K configs: one start per masked block,
# spread over [1/6, 4/6] of training, each lasting 1/6 of training.
num_blocks = cfg["model"]["init_args"]["network"]["init_args"].get("num_blocks", 4)
unit = total_steps / 6
starts = [round(unit + k * 3 * unit / max(1, num_blocks - 1)) for k in range(num_blocks)]
ends = [round(s + unit) for s in starts]
warmup = [min(500, max(1, round(0.05 * total_steps))), min(1000, max(1, round(0.10 * total_steps)))]

model_args = cfg["model"]["init_args"]
model_args.update(
    lr=LR,
    warmup_steps=warmup,
    attn_mask_annealing_start_steps=starts,
    attn_mask_annealing_end_steps=ends,
    tumor_score_thresh=TUMOR_SCORE_THRESH,
)

if INIT_FROM_COCO:
    assert IMG_SIZE == 640, "The COCO EoMT-B checkpoint is trained at 640x640"
    from huggingface_hub import hf_hub_download

    model_args["ckpt_path"] = hf_hub_download(COCO_CKPT_REPO, "pytorch_model.bin")
    model_args["load_ckpt_class_head"] = False  # 133 COCO classes -> 2 BTXRD classes

cfg["data"]["init_args"].update(
    path=str(CACHE_DIR),
    batch_size=BATCH_SIZE_PER_GPU,
    num_workers=NUM_WORKERS,
    img_size=[IMG_SIZE, IMG_SIZE],
    check_empty_targets=False,  # keep normal images: needed to learn tumor detection
    balanced_sampling=BALANCED_SAMPLING,
)

# Loggers: CSV always (read back below); W&B first if a key is available,
# so main.py's `logger.experiment.log_code` keeps working as in the original project.
loggers = [
    {
        "class_path": "lightning.pytorch.loggers.CSVLogger",
        "init_args": {"save_dir": str(LOG_DIR), "name": RUN_NAME},
    }
]
wandb_key = os.environ.get("WANDB_API_KEY")
if ON_CLOUD and USE_WANDB and not wandb_key:
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient

            wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
        else:
            from google.colab import userdata

            wandb_key = userdata.get("WANDB_API_KEY")
    except Exception:
        wandb_key = None
if USE_WANDB and wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
    wandb_logger = cfg["trainer"]["logger"]
    wandb_logger["init_args"].update(name=RUN_NAME, save_dir=str(OUT_DIR))
    loggers.insert(0, wandb_logger)
print("Loggers:", [l["class_path"].rsplit(".", 1)[-1] for l in loggers])

trainer = cfg["trainer"]
trainer.update(
    max_epochs=epochs,
    devices=devices,
    accelerator="gpu" if NUM_GPUS else "cpu",
    precision="16-mixed" if NUM_GPUS else "32-true",
    logger=loggers,
    log_every_n_steps=min(50, max(1, steps_per_epoch)),
)
if devices > 1:
    trainer["strategy"] = "ddp"
for cb in trainer["callbacks"]:
    if cb["class_path"].endswith("ModelCheckpoint"):
        cb["init_args"]["dirpath"] = str(CKPT_DIR)
if SMOKE_TEST:
    trainer.update(limit_train_batches=2, limit_val_batches=2, num_sanity_val_steps=0)
cfg["seed_everything"] = SEED

with open(RUN_CONFIG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"train images={len(data.train_dataset)}, devices={devices}, total batch={BATCH_SIZE_PER_GPU * devices}")
print(f"steps/epoch={steps_per_epoch}, total steps={total_steps}, warmup={warmup}")
print(f"annealing starts={starts} ends={ends}")
print(RUN_CONFIG.read_text())

## 9. Huấn luyện

Chạy đúng lệnh của dự án (`main.py fit`) trong một tiến trình con, để Lightning dùng được DDP khi có 2 GPU (Kaggle). Nếu `CKPT_DIR` đã có `last.ckpt` (lần chạy trước bị ngắt), quá trình huấn luyện tiếp tục từ đó qua `--ckpt_path`. Log được in trực tiếp ra bên dưới; checkpoint tốt nhất (theo `metrics/val_score`) và `last.ckpt` được lưu ở `CKPT_DIR`.

> Kaggle giới hạn 12 giờ mỗi phiên; Colab có thể ngắt sớm hơn (đã có resume). Với khoảng 2.980 ảnh train (gấp đôi khi chỉ dùng ảnh có u), 40 epoch EoMT-B 640 trên 2×T4 là khoảng 14.900 bước, mất cỡ vài giờ. Có thể giảm `EPOCHS` nếu cần.

In [ ]:
def run(cmd):
    """Run a command and stream its output into the notebook."""
    print("$", " ".join(map(str, cmd)), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        list(map(str, cmd)), cwd=REPO_DIR, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")


fit_cmd = [sys.executable, "main.py", "fit", "-c", RUN_CONFIG]
if not COMPILE:
    fit_cmd.append("--compile_disabled")
last_ckpt = CKPT_DIR / "last.ckpt"
if RESUME and not SMOKE_TEST and last_ckpt.exists():
    print("Resuming from", last_ckpt)
    fit_cmd += ["--ckpt_path", last_ckpt]
run(fit_cmd)

## 10. Đường cong huấn luyện và xuất trọng số tốt nhất

Checkpoint Lightning (`.ckpt`) được chuyển thành state dict thuần (`.bin`), cùng định dạng `pytorch_model.bin` mà README dùng cho `--model.ckpt_path`.

In [ ]:
best_ckpts = sorted(CKPT_DIR.glob("best-*.ckpt"), key=os.path.getmtime)
best_ckpt = best_ckpts[-1] if best_ckpts else CKPT_DIR / "last.ckpt"
print("Best checkpoint:", best_ckpt)

ckpt = torch.load(best_ckpt, map_location="cpu", weights_only=False)
state_dict = {k.replace("._orig_mod", ""): v for k, v in ckpt["state_dict"].items()}
torch.save(state_dict, BEST_WEIGHTS)
print(f"Saved {len(state_dict)} tensors -> {BEST_WEIGHTS} (epoch {ckpt.get('epoch')})")
del ckpt

# One CSV per run (a resumed run starts a new version): read them all
metrics_csvs = sorted(LOG_DIR.glob(f"{RUN_NAME}/version_*/metrics.csv"), key=os.path.getmtime)
hist = pd.concat([pd.read_csv(p) for p in metrics_csvs], ignore_index=True)
val = hist.dropna(subset=["metrics/val_score"]).groupby("epoch").last()
loss_col = "losses/train_loss_total"  # logged every `log_every_n_steps` steps
train = hist.dropna(subset=[loss_col]).groupby("epoch")[loss_col].mean() if loss_col in hist else pd.Series(dtype=float)

panels = {
    "Segmentation (val)": ["metrics/val_dice", "metrics/val_iou", "metrics/val_dice_all", "metrics/val_ap_all"],
    "Tumor detection (val)": ["metrics/val_det_auroc", "metrics/val_det_sensitivity", "metrics/val_det_specificity"],
    "Benign / malignant + 3-class (val)": ["metrics/val_cls_auroc", "metrics/val_cls_balanced_acc", "metrics/val_img3_balanced_acc"],
}
fig, axes = plt.subplots(1, 4, figsize=(24, 4.5))
axes[0].plot(train.index, train.values, marker="o")
axes[0].set_title("Train loss (total)")
for ax, (title, cols) in zip(axes[1:], panels.items()):
    for col in cols:
        if col in val:
            ax.plot(val.index, val[col], marker="o", label=col.split("/")[-1].replace("val_", ""))
    ax.set_title(title)
    ax.legend()
for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

summary_cols = ["metrics/val_score", "metrics/val_det_auroc", "metrics/val_dice", "metrics/val_dice_all",
                "metrics/val_cls_auroc", "metrics/val_img3_balanced_acc", "metrics/val_ap_all"]
display(val[[c for c in summary_cols if c in val]].round(4).tail(5))

## 11. Đánh giá trên tập test bằng CLI của dự án

Giống phần *Evaluating* trong README: tắt masked attention khi suy luận (`masked_attn_enabled: False`) và nạp trọng số tốt nhất qua `model.ckpt_path`. Chạy trên 1 GPU để không bị lặp mẫu do DDP.

In [ ]:
test_cfg = yaml.safe_load(open(RUN_CONFIG))
test_cfg["model"]["init_args"].update(ckpt_path=str(BEST_WEIGHTS), load_ckpt_class_head=True)
test_cfg["model"]["init_args"]["network"]["init_args"]["masked_attn_enabled"] = False
test_cfg["trainer"].update(
    devices=1,
    logger=[{"class_path": "lightning.pytorch.loggers.CSVLogger",
             "init_args": {"save_dir": str(LOG_DIR), "name": f"{RUN_NAME}_test"}}],
)
test_cfg["trainer"].pop("strategy", None)
for key in ("limit_train_batches", "limit_val_batches"):
    test_cfg["trainer"].pop(key, None)
with open(TEST_CONFIG, "w") as f:
    yaml.safe_dump(test_cfg, f, sort_keys=False)

has_test = (CACHE_DIR / "test" / "images").is_dir()
run([sys.executable, "main.py", "test" if has_test else "validate", "-c", TEST_CONFIG])

## 12. Phân tích chi tiết và trực quan hóa trên tập test

Nạp mô hình trong notebook theo cách của `inference.ipynb`, dự đoán từng ảnh bằng chính hàm `MaskClassificationBTXRD.predict_btxrd` dùng khi tính metric: **một lần forward** cho ra cùng lúc có u hay không, vùng u và lành / ác. Sau đó đánh giá riêng từng phần: phát hiện, phân đoạn, phân loại lành / ác và phân loại 3 lớp, rồi xem các ca bỏ sót, báo động nhầm và Dice thấp.

In [ ]:
import importlib
import warnings
import torch.nn.functional as F
from torch.amp import autocast

warnings.filterwarnings("ignore", message=r".*is an instance of `nn\.Module` and is already saved.*")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_model(config, weights):
    net_cfg = config["model"]["init_args"]["network"]
    enc_cfg = net_cfg["init_args"]["encoder"]
    img_size = tuple(config["data"]["init_args"]["img_size"])
    num_classes = config["data"]["init_args"].get("num_classes", 2)

    def load_cls(path):
        module, name = path.rsplit(".", 1)
        return getattr(importlib.import_module(module), name)

    encoder = load_cls(enc_cfg["class_path"])(
        img_size=img_size, ckpt_path=str(weights), **enc_cfg.get("init_args", {})
    )
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k not in ("encoder", "masked_attn_enabled")}
    network = load_cls(net_cfg["class_path"])(
        encoder=encoder, num_classes=num_classes, masked_attn_enabled=False, **net_kwargs
    )
    model_kwargs = {
        k: v for k, v in config["model"]["init_args"].items()
        if k not in ("network", "ckpt_path", "load_ckpt_class_head")
    }
    model = load_cls(config["model"]["class_path"])(
        network=network, img_size=img_size, num_classes=num_classes,
        ckpt_path=str(weights), load_ckpt_class_head=True, **model_kwargs,
    )
    return model.eval().to(device)


model = build_model(test_cfg, BEST_WEIGHTS)


@torch.no_grad()
def predict(img):
    """img: uint8 tensor (3, H, W) -> tumor mask (H, W), P(tumor), P(malignant), class 0/1/2."""
    with autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
        imgs = [img.to(device)]
        sizes = [img.shape[-2:]]
        x = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(x)
        mask_logits = F.interpolate(mask_logits_per_layer[-1], model.img_size, mode="bilinear")
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(mask_logits, sizes)[0]
        mask, p_tumor, p_malignant, image_pred = model.predict_btxrd(
            mask_logits.float(), class_logits_per_layer[-1][0],
            model.tumor_score_thresh, model.malignant_thresh,
        )
    return mask.cpu(), float(p_tumor), float(p_malignant), int(image_pred)


eval_ds = data.test_dataset if data.test_dataset is not None else data.val_dataset
records, cache = [], {}
for idx in range(len(eval_ds)):
    img, tgt = eval_ds[idx]
    mask, p_tumor, p_mal, pred = predict(img)
    gt = tgt["masks"].bool().any(0)
    label = int(tgt["image_label"])
    inter = (mask & gt).sum().item()
    has_gt_mask = bool(gt.any())
    records.append(
        dict(
            image=eval_ds.samples[idx][0].stem,
            label=label,  # 0 normal, 1 benign, 2 malignant
            pred=pred,
            has_tumor=int(label > 0),
            p_tumor=p_tumor,
            pred_tumor=int(pred > 0),
            p_malignant=p_mal,
            dice=2 * inter / max(1, mask.sum().item() + gt.sum().item()) if has_gt_mask else np.nan,
            iou=inter / max(1, (mask | gt).sum().item()) if has_gt_mask else np.nan,
            pred_area=mask.float().mean().item(),
        )
    )
    cache[idx] = mask

pred_df = pd.DataFrame(records)
pred_df["label_name"] = pred_df.label.map(dict(enumerate(IMAGE_CLASSES)))
pred_df["pred_name"] = pred_df.pred.map(dict(enumerate(IMAGE_CLASSES)))
pred_df.to_csv(OUT_DIR / f"{RUN_NAME}_test_predictions.csv", index=False)

tumor = pred_df[pred_df.has_tumor == 1]
normal = pred_df[pred_df.has_tumor == 0]
print(f"{len(pred_df)} images ({len(normal)} normal, {len(tumor)} tumor)")
print(f"Detection    | sensitivity {(tumor.pred_tumor == 1).mean():.4f} | specificity {(normal.pred_tumor == 0).mean():.4f}")
print(f"Segmentation | Dice {tumor.dice.mean():.4f} | IoU {tumor.iou.mean():.4f} (tumor images)")
display(pred_df.groupby("label_name")[["p_tumor", "pred_tumor", "dice", "iou", "pred_area"]].mean().round(4))

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
    roc_auc_score,
)

print("=== 1. Tumor detection (all images) ===")
print(classification_report(pred_df.has_tumor, pred_df.pred_tumor, labels=[0, 1],
                            target_names=["normal", "tumor"], digits=4, zero_division=0))
if pred_df.has_tumor.nunique() == 2:
    print(f"AUROC: {roc_auc_score(pred_df.has_tumor, pred_df.p_tumor):.4f}\n")

print("=== 2. Benign / malignant (tumor images, top query) ===")
is_mal = (tumor.label == 2).astype(int)
print(classification_report(is_mal, (tumor.p_malignant >= 0.5).astype(int), labels=[0, 1],
                            target_names=["benign", "malignant"], digits=4, zero_division=0))
if is_mal.nunique() == 2:
    print(f"AUROC: {roc_auc_score(is_mal, tumor.p_malignant):.4f}\n")

print("=== 3. Final 3-class output (normal / benign / malignant) ===")
print(classification_report(pred_df.label, pred_df.pred, labels=[0, 1, 2],
                            target_names=list(IMAGE_CLASSES), digits=4, zero_division=0))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
ConfusionMatrixDisplay.from_predictions(
    pred_df.label, pred_df.pred, labels=[0, 1, 2], display_labels=IMAGE_CLASSES, cmap="Blues", ax=axes[0, 0]
)
axes[0, 0].set_title("Confusion matrix (3 classes)")
if pred_df.has_tumor.nunique() == 2:
    RocCurveDisplay.from_predictions(pred_df.has_tumor, pred_df.p_tumor, name="EoMT", ax=axes[0, 1])
    axes[0, 1].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0, 1].set_title("ROC tumor detection (tumor = positive)")
if is_mal.nunique() == 2:
    RocCurveDisplay.from_predictions(is_mal, tumor.p_malignant, name="EoMT", ax=axes[0, 2])
    axes[0, 2].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0, 2].set_title("ROC benign / malignant (malignant = positive)")

for name, g in pred_df.groupby("label_name"):
    axes[1, 0].hist(g.p_tumor, bins=20, range=(0, 1), alpha=0.6, label=name)
axes[1, 0].axvline(TUMOR_SCORE_THRESH, color="k", ls="--")
axes[1, 0].set_title("Tumor score per image")
axes[1, 0].legend()
for c in (1, 2):
    axes[1, 1].hist(tumor.dice[tumor.label == c], bins=20, range=(0, 1), alpha=0.6, label=IMAGE_CLASSES[c])
axes[1, 1].set_title("Tumor Dice per image (tumor images)")
axes[1, 1].set_xlabel("Dice")
axes[1, 1].legend()
axes[1, 2].hist(normal.pred_area, bins=20, alpha=0.8, color="tab:gray")
axes[1, 2].set_title("Predicted tumor area on normal images (fraction of image)")
plt.tight_layout()
plt.show()

In [ ]:
def show_predictions(indices, title):
    if len(indices) == 0:
        return
    fig, axes = plt.subplots(len(indices), 3, figsize=(13, 4.3 * len(indices)), squeeze=False)
    for row, idx in zip(axes, indices):
        img, tgt = eval_ds[idx]
        r = pred_df.iloc[idx]
        base = img.permute(1, 2, 0).numpy()
        gt = tgt["masks"].bool().any(0).numpy()
        pred = cache[idx].numpy()

        row[0].imshow(base)
        row[0].set_title(r.image, fontsize=9)
        dice = "" if np.isnan(r.dice) else f" Dice={r.dice:.2f}"
        for ax, m, cls, txt in (
            (row[1], gt, r.label, f"Ground truth: {r.label_name}"),
            (row[2], pred, r.pred,
             f"Prediction: {r.pred_name} (P_tumor={r.p_tumor:.2f}, P_mal={r.p_malignant:.2f}){dice}"),
        ):
            ax.imshow(base)
            if cls > 0 and m.any():
                rgba = np.zeros((*m.shape, 4))
                rgba[m] = [*MASK_COLORS[int(cls) - 1], 0.45]
                ax.imshow(rgba)
            if gt.any():
                ax.contour(gt, levels=[0.5], colors="yellow", linewidths=1)
            ax.set_title(txt, fontsize=9)
        for ax in row:
            ax.axis("off")
    fig.suptitle(title + "  (green = benign, red = malignant, yellow contour = ground truth)", y=1.0)
    plt.tight_layout()
    plt.show()


rng = np.random.default_rng(SEED)
show_predictions(rng.choice(len(pred_df), size=min(4, len(pred_df)), replace=False), "Random test images")

missed = pred_df.index[(pred_df.has_tumor == 1) & (pred_df.pred_tumor == 0)].tolist()
false_alarm = pred_df.index[(pred_df.has_tumor == 0) & (pred_df.pred_tumor == 1)].tolist()
swapped = pred_df.index[(pred_df.label > 0) & (pred_df.pred > 0) & (pred_df.label != pred_df.pred)].tolist()
show_predictions(missed[:3], "Missed tumors (tumor -> predicted normal)")
show_predictions(false_alarm[:3], "False alarms (normal -> predicted tumor)")
show_predictions(swapped[:3], "Benign <-> malignant swapped")
show_predictions(tumor.nsmallest(min(3, len(tumor)), "dice").index.tolist(), "Lowest segmentation Dice")

## 13. Dự đoán cho một ảnh X-quang mới

In [ ]:
def predict_file(path, max_side=CACHE_MAX_SIDE):
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.thumbnail((max_side, max_side), Image.LANCZOS)
        img = torch.from_numpy(np.array(im)).permute(2, 0, 1).contiguous()
    mask, p_tumor, p_mal, pred = predict(img)
    label = IMAGE_CLASSES[pred]

    plt.figure(figsize=(6, 6))
    plt.imshow(img.permute(1, 2, 0).numpy())
    if pred > 0:
        rgba = np.zeros((*mask.shape, 4))
        rgba[mask.numpy()] = [*MASK_COLORS[pred - 1], 0.45]
        plt.imshow(rgba)
        title = f"{Path(path).name}: {label} tumor (P_tumor={p_tumor:.3f}, P_malignant={p_mal:.3f})"
    else:
        title = f"{Path(path).name}: no tumor detected (P_tumor={p_tumor:.3f})"
    plt.title(title, fontsize=10)
    plt.axis("off")
    plt.show()
    return label, p_tumor, p_mal, mask


ids = eval_ds.class_ids
for c in (0, 1, 2):  # one normal, one benign and one malignant image
    if c in ids:
        print(predict_file(eval_ds.samples[ids.index(c)][0])[:3])

## 14. Kết quả đầu ra

Các file trong `OUT_DIR`: `MyDrive/btxrd_eomt/` trên Colab, `/kaggle/working` trên Kaggle (tải về ở tab *Output* sau khi *Save Version*):

| File | Nội dung |
|---|---|
| `btxrd_instance_eomt_base_640_best.bin` | Trọng số tốt nhất (state dict), dùng với `--model.ckpt_path` |
| `checkpoints/btxrd_instance_eomt_base_640/*.ckpt` | Checkpoint Lightning (best + last, có thể resume) |
| `btxrd_instance_eomt_base_640.yaml` | Cấu hình đầy đủ của lần chạy |
| `btxrd_instance_eomt_base_640_test_predictions.csv` | Dự đoán theo ảnh trên tập test: nhãn thật, P(u), P(ác), lớp dự đoán, Dice |
| `logs/` | Log CSV (và `wandb/` nếu dùng W&B) |

Đánh giá lại ở máy khác bằng CLI của dự án (dữ liệu đã chia bằng `tools/split_btxrd.py`):
```bash
python3 tools/split_btxrd.py --src /path/to/BTXRD --dst /path/to/BTXRD_full_split
python3 main.py test -c btxrd_instance_eomt_base_640_test.yaml \
  --data.path /path/to/BTXRD_full_split \
  --model.ckpt_path btxrd_instance_eomt_base_640_best.bin
```

In [ ]:
for p in sorted(OUT_DIR.iterdir()):
    print(p.name)
# Free up disk / Kaggle output size: the image cache can be rebuilt from the input dataset
shutil.rmtree(CACHE_DIR, ignore_errors=True)